# Module 4 • Distributional Semantics and Word Embeddings

# Lesson 23 • GloVe, FastText, and Subword-Aware Embeddings

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 120–150 minutes

---

## Scope

This lesson compares GloVe-style global co-occurrence embeddings with
FastText-style subword-aware embeddings. It includes self-contained NumPy
implementations, rare-word and out-of-vocabulary analysis, evaluation, and Arabic
morphology examples.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain how GloVe differs from Word2Vec;
- build a weighted co-occurrence matrix;
- explain and implement the GloVe objective;
- inspect GloVe nearest neighbors;
- generate character n-grams with boundary symbols;
- compose known and unseen word vectors from subwords;
- implement a toy FastText-style Skip-Gram model;
- compare word-only and subword-aware embeddings;
- evaluate rare-word and OOV behavior;
- discuss Arabic morphology and multilingual limitations.

## Table of Contents

1. Embedding Families
2. Global and Local Training Signals
3. Corpus and Vocabulary
4. Weighted Co-Occurrence Matrix
5. GloVe Objective
6. GloVe Weighting Function
7. NumPy GloVe Training
8. GloVe Nearest Neighbors
9. GloVe Strengths and Limitations
10. Why Subwords Matter
11. Character n-Grams
12. Boundary Symbols
13. Subword Vocabulary
14. FastText Intuition
15. OOV Vector Construction
16. Subword Similarity
17. Toy Subword Skip-Gram
18. Comparing Embeddings
19. Rare Words and Morphology
20. Hyperparameters
21. Evaluation
22. Bias and Responsible Use
23. Arabic and Multilingual Considerations
24. Reproducibility
25. Knowledge Check
26. Exercises
27. Summary and Next Lesson

# 1. Embedding Families

| Method | Signal | Representation |
|---|---|---|
| PPMI + SVD | explicit co-occurrence | word |
| Word2Vec | local prediction | word |
| GloVe | global co-occurrence reconstruction | word |
| FastText | local prediction | word + character n-grams |

In [ ]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

embedding_families = pd.DataFrame(
    [
        ("PPMI + SVD", "count-based", "global counts", False),
        ("Word2Vec", "predictive", "local context", False),
        ("GloVe", "weighted factorization", "global co-occurrence", False),
        ("FastText", "predictive", "local context + subwords", True),
    ],
    columns=["Method", "Family", "Signal", "Subword aware"],
)
embedding_families

# 2. Global and Local Training Signals

Word2Vec learns from individual center-context events. GloVe aggregates those
events into a global co-occurrence matrix and learns vectors that reconstruct
logarithmic co-occurrence values.

In [ ]:
pd.DataFrame(
    [
        ("Local", "one center-context event", "Word2Vec"),
        ("Global", "aggregated word-context counts", "GloVe"),
    ],
    columns=["Perspective", "Training unit", "Example"],
)

# 3. Corpus and Vocabulary

In [ ]:
corpus = [
    "doctor treats patient in hospital",
    "nurse cares for patient in clinic",
    "physician examines patient in hospital",
    "surgeon works in hospital",
    "teacher teaches student in school",
    "professor teaches student at university",
    "researcher works at university",
    "student studies lesson in school",
    "driver operates vehicle on road",
    "pilot operates aircraft at airport",
    "traveler boards aircraft at airport",
    "passenger rides vehicle on road",
    "doctor and nurse work together",
    "teacher and professor work together",
    "hospital employs doctor and nurse",
    "university employs professor and researcher",
]

tokenized_corpus = [sentence.lower().split() for sentence in corpus]
token_counts = Counter(
    token
    for sentence in tokenized_corpus
    for token in sentence
)

vocabulary = sorted(token_counts)
word_to_index = {word: i for i, word in enumerate(vocabulary)}
index_to_word = {i: word for word, i in word_to_index.items()}

print("Sentences:", len(corpus))
print("Vocabulary size:", len(vocabulary))

# 4. Weighted Co-Occurrence Matrix

Contexts closer to the target can receive larger weights. A common educational
choice is inverse-distance weighting:

\[
weight = 1 / distance
\]

In [ ]:
def build_weighted_cooccurrence(
    sentences,
    mapping,
    window_size=3,
):
    matrix = np.zeros(
        (len(mapping), len(mapping)),
        dtype=float,
    )

    for sentence in sentences:
        for center_pos, center_word in enumerate(sentence):
            center_id = mapping[center_word]
            left = max(0, center_pos - window_size)
            right = min(len(sentence), center_pos + window_size + 1)

            for context_pos in range(left, right):
                if context_pos == center_pos:
                    continue

                context_id = mapping[sentence[context_pos]]
                distance = abs(center_pos - context_pos)
                matrix[center_id, context_id] += 1.0 / distance

    return matrix


cooccurrence = build_weighted_cooccurrence(
    tokenized_corpus,
    word_to_index,
    window_size=3,
)

cooccurrence_frame = pd.DataFrame(
    cooccurrence,
    index=vocabulary,
    columns=vocabulary,
)

cooccurrence_frame.loc[
    ["doctor", "nurse", "teacher", "professor"],
    ["patient", "hospital", "student", "school", "university", "work"],
].round(2)

# 5. GloVe Objective

GloVe learns word and context vectors whose dot products approximate the logarithm
of observed co-occurrence counts.

\[
J = \sum_{i,j} f(X_{ij})
\left(w_i^T \widetilde{w}_j + b_i + \widetilde{b}_j - \log X_{ij}ight)^2
\]

The logarithm compresses large count differences. The weighting function reduces
the influence of noisy rare pairs and prevents extremely frequent pairs from
dominating.

# 6. GloVe Weighting Function

In [ ]:
def glove_weight(value, x_max=10.0, alpha=0.75):
    if value <= 0:
        return 0.0
    if value < x_max:
        return (value / x_max) ** alpha
    return 1.0


weight_demo = pd.DataFrame(
    {"cooccurrence": [0, 0.5, 1, 2, 5, 10, 20]}
)
weight_demo["weight"] = weight_demo["cooccurrence"].map(glove_weight)
weight_demo

# 7. NumPy GloVe Training

In [ ]:
def train_glove(
    matrix,
    dimension=16,
    epochs=160,
    learning_rate=0.04,
    x_max=10.0,
    alpha=0.75,
    seed=42,
):
    generator = np.random.default_rng(seed)
    vocab_size = matrix.shape[0]

    word_matrix = generator.normal(
        0.0, 0.1, size=(vocab_size, dimension)
    )
    context_matrix = generator.normal(
        0.0, 0.1, size=(vocab_size, dimension)
    )
    word_bias = np.zeros(vocab_size)
    context_bias = np.zeros(vocab_size)

    nonzero_pairs = np.argwhere(matrix > 0)
    losses = []

    for epoch in range(epochs):
        generator.shuffle(nonzero_pairs)
        total_loss = 0.0
        current_rate = learning_rate * (
            1.0 - 0.7 * epoch / max(epochs - 1, 1)
        )

        for word_id, context_id in nonzero_pairs:
            count = matrix[word_id, context_id]
            weight = glove_weight(count, x_max=x_max, alpha=alpha)

            word_vector = word_matrix[word_id].copy()
            context_vector = context_matrix[context_id].copy()

            error = (
                np.dot(word_vector, context_vector)
                + word_bias[word_id]
                + context_bias[context_id]
                - math.log(count)
            )

            weighted_error = weight * error
            total_loss += weight * error ** 2

            word_matrix[word_id] -= (
                current_rate * weighted_error * context_vector
            )
            context_matrix[context_id] -= (
                current_rate * weighted_error * word_vector
            )
            word_bias[word_id] -= current_rate * weighted_error
            context_bias[context_id] -= current_rate * weighted_error

        losses.append(total_loss / len(nonzero_pairs))

    return (
        word_matrix,
        context_matrix,
        word_bias,
        context_bias,
        losses,
    )

In [ ]:
(
    glove_word_vectors,
    glove_context_vectors,
    glove_word_biases,
    glove_context_biases,
    glove_losses,
) = train_glove(cooccurrence)

glove_embeddings = glove_word_vectors + glove_context_vectors

print("Initial loss:", round(glove_losses[0], 4))
print("Final loss:", round(glove_losses[-1], 4))
print("Embedding shape:", glove_embeddings.shape)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(glove_losses)
plt.title("GloVe-Style Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Average weighted squared error")
plt.tight_layout()
plt.show()

Lower training loss indicates better reconstruction of the observed matrix, not
necessarily better semantic quality.

# 8. GloVe Nearest Neighbors

In [ ]:
def nearest_neighbors(
    target_word,
    embeddings,
    mapping,
    reverse_mapping,
    top_k=6,
):
    target_id = mapping[target_word]
    scores = cosine_similarity(
        embeddings[target_id].reshape(1, -1),
        embeddings,
    ).ravel()

    rows = []
    for word_id, score in enumerate(scores):
        word = reverse_mapping[word_id]
        if word == target_word:
            continue
        rows.append(
            {"word": word, "similarity": float(score)}
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["similarity", "word"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


nearest_neighbors(
    "doctor",
    glove_embeddings,
    word_to_index,
    index_to_word,
)

In [ ]:
for target in [
    "doctor",
    "teacher",
    "hospital",
    "university",
    "vehicle",
    "aircraft",
]:
    print(target)
    display(
        nearest_neighbors(
            target,
            glove_embeddings,
            word_to_index,
            index_to_word,
            top_k=4,
        )
    )

# 9. GloVe Strengths and Limitations

**Strengths**

- uses global co-occurrence information;
- connects matrix factorization and embeddings;
- produces reusable static vectors.

**Limitations**

- one vector per word type;
- no native OOV vector;
- morphology is not explicitly modeled;
- results inherit corpus bias.

# 10. Why Subwords Matter

Word-only embeddings treat related forms as separate vocabulary items:

```text
teach, teacher, teaches, teaching
```

Subword-aware models share information through character patterns.

In [ ]:
pd.DataFrame(
    [
        ("teach", "base"),
        ("teacher", "agent noun"),
        ("teaches", "third-person singular"),
        ("teaching", "progressive or verbal noun"),
    ],
    columns=["Word", "Morphological relation"],
)

# 11. Character n-Grams

In [ ]:
def character_ngrams(
    word,
    min_n=3,
    max_n=5,
    add_boundaries=True,
):
    if min_n <= 0 or max_n < min_n:
        raise ValueError("Require 0 < min_n <= max_n")

    marked = f"<{word}>" if add_boundaries else word
    ngrams = []

    for n in range(min_n, max_n + 1):
        for start in range(len(marked) - n + 1):
            ngrams.append(marked[start:start + n])

    return sorted(set(ngrams))


character_ngrams(
    "teacher",
    min_n=3,
    max_n=4,
)[:20]

# 12. Boundary Symbols

Boundary symbols distinguish prefixes and suffixes:

```text
<tea
her>
```

FastText also includes a representation for the complete word.

In [ ]:
print(
    "With boundaries:",
    character_ngrams("cat", 3, 3, True),
)
print(
    "Without boundaries:",
    character_ngrams("cat", 3, 3, False),
)

# 13. Subword Vocabulary

In [ ]:
def build_subword_vocabulary(
    words,
    min_n=3,
    max_n=5,
):
    subword_set = set()
    word_to_subwords = {}

    for word in words:
        ngrams = character_ngrams(
            word,
            min_n=min_n,
            max_n=max_n,
        )
        ngrams.append(f"<WORD:{word}>")
        unique = sorted(set(ngrams))
        word_to_subwords[word] = unique
        subword_set.update(unique)

    subwords = sorted(subword_set)
    subword_to_index = {
        subword: i
        for i, subword in enumerate(subwords)
    }

    word_to_subword_ids = {
        word: [
            subword_to_index[subword]
            for subword in word_subwords
        ]
        for word, word_subwords in word_to_subwords.items()
    }

    return (
        subwords,
        subword_to_index,
        word_to_subword_ids,
    )


(
    subword_vocabulary,
    subword_to_index,
    word_to_subword_ids,
) = build_subword_vocabulary(vocabulary)

print("Word vocabulary:", len(vocabulary))
print("Subword vocabulary:", len(subword_vocabulary))

Production FastText commonly hashes n-grams into a fixed number of buckets instead
of storing each n-gram explicitly.

# 14. FastText Intuition

FastText extends Skip-Gram or CBOW by constructing a word from its subword vectors.

\[
v_{word} = rac{1}{|G(word)|}
\sum_{g \in G(word)} z_g
\]

The context-prediction objective remains similar to Word2Vec. The center-word
representation is the main difference.

# 15. OOV Vector Construction

In [ ]:
random_generator = np.random.default_rng(42)
subword_dimension = 16

random_subword_vectors = random_generator.normal(
    0.0,
    0.1,
    size=(len(subword_vocabulary), subword_dimension),
)


def compose_known_word_vector(word, subword_matrix):
    ids = word_to_subword_ids[word]
    return subword_matrix[ids].mean(axis=0)


def compose_oov_vector(
    word,
    subword_matrix,
    min_n=3,
    max_n=5,
):
    ngrams = character_ngrams(
        word,
        min_n=min_n,
        max_n=max_n,
    )

    known = [
        ngram
        for ngram in ngrams
        if ngram in subword_to_index
    ]

    if not known:
        return None, []

    ids = [subword_to_index[ngram] for ngram in known]
    return subword_matrix[ids].mean(axis=0), known


oov_vector, matched = compose_oov_vector(
    "teachers",
    random_subword_vectors,
)

print("Vector available:", oov_vector is not None)
print("Matched n-grams:", matched[:12])

OOV vectors are approximate. Similar spelling does not guarantee similar meaning.

# 16. Subword Similarity

In [ ]:
def jaccard_subword_similarity(
    word_a,
    word_b,
    min_n=3,
    max_n=5,
):
    set_a = set(
        character_ngrams(word_a, min_n, max_n)
    )
    set_b = set(
        character_ngrams(word_b, min_n, max_n)
    )

    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0


pairs = [
    ("teach", "teacher"),
    ("teacher", "teachers"),
    ("doctor", "doctors"),
    ("doctor", "hospital"),
    ("vehicle", "vehicles"),
]

for left, right in pairs:
    print(
        f"{left:<10} {right:<10} "
        f"{jaccard_subword_similarity(left, right):.3f}"
    )

# 17. Toy Subword Skip-Gram

In [ ]:
def build_skipgram_pairs(
    sentences,
    mapping,
    window_size=2,
):
    pairs = []

    for sentence in sentences:
        for center_pos, center_word in enumerate(sentence):
            center_id = mapping[center_word]
            left = max(0, center_pos - window_size)
            right = min(len(sentence), center_pos + window_size + 1)

            for context_pos in range(left, right):
                if context_pos == center_pos:
                    continue

                context_id = mapping[sentence[context_pos]]
                pairs.append((center_id, context_id))

    return pairs


skipgram_pairs = build_skipgram_pairs(
    tokenized_corpus,
    word_to_index,
)

frequencies = np.array(
    [token_counts[word] for word in vocabulary],
    dtype=float,
)
negative_distribution = frequencies ** 0.75
negative_distribution /= negative_distribution.sum()

print("Training pairs:", len(skipgram_pairs))

In [ ]:
def sigmoid(value):
    return 1.0 / (
        1.0 + np.exp(-np.clip(value, -20, 20))
    )


def sample_negative_ids(
    count,
    positive_id,
    distribution,
    generator,
):
    negatives = []

    while len(negatives) < count:
        candidate = int(
            generator.choice(
                len(distribution),
                p=distribution,
            )
        )

        if candidate != positive_id:
            negatives.append(candidate)

    return negatives

In [ ]:
def train_subword_skipgram(
    pairs,
    epochs=100,
    learning_rate=0.04,
    negatives_per_positive=5,
    dimension=16,
    seed=42,
):
    generator = np.random.default_rng(seed)

    subword_matrix = generator.normal(
        0.0,
        0.1,
        size=(len(subword_vocabulary), dimension),
    )
    output_matrix = generator.normal(
        0.0,
        0.1,
        size=(len(vocabulary), dimension),
    )

    pair_array = np.array(pairs, dtype=int)
    losses = []

    for epoch in range(epochs):
        generator.shuffle(pair_array)
        total_loss = 0.0
        current_rate = learning_rate * (
            1.0 - 0.7 * epoch / max(epochs - 1, 1)
        )

        for center_id, positive_id in pair_array:
            center_word = index_to_word[int(center_id)]
            subword_ids = word_to_subword_ids[center_word]

            center_vector = subword_matrix[
                subword_ids
            ].mean(axis=0)

            positive_vector = output_matrix[
                positive_id
            ].copy()

            positive_score = float(
                sigmoid(
                    np.dot(
                        center_vector,
                        positive_vector,
                    )
                )
            )

            positive_error = positive_score - 1.0
            total_loss -= math.log(
                max(positive_score, 1e-12)
            )

            center_gradient = (
                positive_error * positive_vector
            )

            output_matrix[positive_id] -= (
                current_rate
                * positive_error
                * center_vector
            )

            negative_ids = sample_negative_ids(
                negatives_per_positive,
                int(positive_id),
                negative_distribution,
                generator,
            )

            for negative_id in negative_ids:
                negative_vector = output_matrix[
                    negative_id
                ].copy()

                negative_score = float(
                    sigmoid(
                        np.dot(
                            center_vector,
                            negative_vector,
                        )
                    )
                )

                negative_error = negative_score
                total_loss -= math.log(
                    max(
                        1.0 - negative_score,
                        1e-12,
                    )
                )

                center_gradient += (
                    negative_error
                    * negative_vector
                )

                output_matrix[negative_id] -= (
                    current_rate
                    * negative_error
                    * center_vector
                )

            subword_gradient = (
                center_gradient
                / len(subword_ids)
            )

            for subword_id in subword_ids:
                subword_matrix[subword_id] -= (
                    current_rate * subword_gradient
                )

        losses.append(
            total_loss / len(pair_array)
        )

    return subword_matrix, output_matrix, losses

In [ ]:
(
    trained_subword_vectors,
    trained_output_vectors,
    subword_losses,
) = train_subword_skipgram(skipgram_pairs)

trained_word_vectors = np.vstack(
    [
        trained_subword_vectors[
            word_to_subword_ids[word]
        ].mean(axis=0)
        for word in vocabulary
    ]
)

print("Initial loss:", round(subword_losses[0], 4))
print("Final loss:", round(subword_losses[-1], 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(subword_losses)
plt.title("Subword-Aware Skip-Gram Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Average loss")
plt.tight_layout()
plt.show()

# 18. Comparing Embeddings

In [ ]:
comparison_rows = []

for target in [
    "doctor",
    "teacher",
    "vehicle",
    "aircraft",
]:
    glove_neighbors = nearest_neighbors(
        target,
        glove_embeddings,
        word_to_index,
        index_to_word,
        top_k=3,
    )["word"].tolist()

    subword_neighbors = nearest_neighbors(
        target,
        trained_word_vectors,
        word_to_index,
        index_to_word,
        top_k=3,
    )["word"].tolist()

    comparison_rows.append(
        {
            "target": target,
            "glove_neighbors": ", ".join(glove_neighbors),
            "subword_neighbors": ", ".join(subword_neighbors),
        }
    )

pd.DataFrame(comparison_rows)

Differences reflect objective, corpus, initialization, and representation design.

# 19. Rare Words and Morphology

Subword sharing can strengthen rare forms, but character information cannot replace
missing semantic context.

In [ ]:
pd.DataFrame(
    [
        (
            "Word-only",
            "few direct updates",
            "unstable rare-word vector",
        ),
        (
            "Subword-aware",
            "shares character n-grams",
            "better morphological generalization",
        ),
    ],
    columns=[
        "Model",
        "Rare-word behavior",
        "Likely result",
    ],
)

Character overlap can also produce false similarity. For example, `universe` and
`university` share many characters but are not simple morphological variants.

# 20. Hyperparameters

## GloVe

- window size;
- dimension;
- `x_max`;
- `alpha`;
- learning rate;
- epochs.

## FastText-style

- Skip-Gram or CBOW;
- minimum and maximum n-gram length;
- bucket count;
- negative samples;
- minimum frequency;
- dimension.

In [ ]:
pd.DataFrame(
    [
        ("GloVe x_max", "weight saturation"),
        ("GloVe alpha", "low-count weighting curve"),
        ("Subword min_n", "shortest character pattern"),
        ("Subword max_n", "longest character pattern"),
        ("Bucket count", "memory and hash collisions"),
        ("Dimension", "capacity and storage"),
    ],
    columns=["Hyperparameter", "Main effect"],
)

# 21. Evaluation

**Intrinsic evaluation**

- similarity correlation;
- analogies;
- nearest neighbors;
- rare-word benchmarks;
- OOV coverage;
- morphological similarity.

**Extrinsic evaluation**

- classification;
- NER;
- retrieval;
- machine translation;
- morphological tagging.

Training loss should not be reported as the only measure of semantic quality.

# 22. Bias and Responsible Use

GloVe and FastText inherit social and domain associations from training data.
Subword sharing can propagate associations across related names and forms.

In [ ]:
pd.DataFrame(
    [
        ("Corpus composition", "Which groups and varieties are represented?"),
        ("Nearest neighbors", "Do names receive stereotyped associations?"),
        ("OOV coverage", "Which communities receive weaker representations?"),
        ("Downstream errors", "Do failures differ across groups?"),
        ("Documentation", "Are training data and limits reported?"),
    ],
    columns=["Audit area", "Question"],
)

# 23. Arabic and Multilingual Considerations

Arabic benefits from subword modeling because of:

- clitics;
- inflection;
- derivation;
- diacritics;
- orthographic variants;
- dialect diversity.

In [ ]:
arabic_forms = [
    "كتاب",
    "الكتاب",
    "والكتاب",
    "بالكتاب",
    "كتابه",
    "كتب",
    "كاتب",
    "مكتبة",
]

pd.Series(
    {
        word: len(
            character_ngrams(
                word,
                min_n=2,
                max_n=4,
            )
        )
        for word in arabic_forms
    },
    name="Number of 2-4 character n-grams",
)

In [ ]:
arabic_pairs = [
    ("كتاب", "الكتاب"),
    ("كتاب", "والكتاب"),
    ("كتاب", "كتابه"),
    ("كتاب", "مكتبة"),
    ("كتب", "كاتب"),
]

pd.DataFrame(
    [
        {
            "word_1": left,
            "word_2": right,
            "jaccard_2_4": jaccard_subword_similarity(
                left,
                right,
                min_n=2,
                max_n=4,
            ),
        }
        for left, right in arabic_pairs
    ]
)

Character n-grams connect surface forms but do not explicitly identify Arabic
morphemes. Tokenization and morphological segmentation remain important.

Important Arabic decisions include:

- preserving or removing diacritics;
- normalizing Alef and Ya forms;
- segmenting clitics;
- separating MSA and dialect;
- handling Arabizi;
- selecting n-gram lengths.

# 24. Reproducibility

In [ ]:
import platform

metadata = pd.Series(
    {
        "corpus_sentences": len(corpus),
        "vocabulary_size": len(vocabulary),
        "glove_dimension": glove_embeddings.shape[1],
        "glove_window": 3,
        "subword_dimension": trained_word_vectors.shape[1],
        "subword_min_n": 3,
        "subword_max_n": 5,
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
    },
    name="Embedding experiment",
)

metadata

Report corpus source, preprocessing, vocabulary thresholds, context definition,
dimensions, weighting parameters, n-gram range, random seed, and evaluation tasks.

# 25. Knowledge Check

1. How does GloVe differ from Word2Vec?
2. What does the GloVe objective reconstruct?
3. Why are logarithmic co-occurrence values used?
4. What is the purpose of the GloVe weighting function?
5. Why learn word and context vectors?
6. What problem do subword embeddings address?
7. Why add boundary symbols?
8. How does FastText represent a word?
9. How can FastText create an OOV vector?
10. Why is that vector only approximate?
11. How can character overlap be misleading?
12. Why are rare words difficult for word-only models?
13. How do intrinsic and extrinsic evaluation differ?
14. Why should embeddings be audited for bias?
15. Why is subword modeling useful for Arabic?

# 26. Exercises

## Exercise 1 — Co-Occurrence Weighting

Compare uniform and inverse-distance context weighting.

## Exercise 2 — GloVe Update

Implement one manual gradient update for a single co-occurrence pair.

## Exercise 3 — GloVe Hyperparameters

Compare several `x_max`, `alpha`, and dimension values.

## Exercise 4 — Character n-Grams

Generate n-grams with and without boundary symbols.

## Exercise 5 — OOV Coverage

Test inflected, misspelled, and domain-specific unseen words.

## Exercise 6 — Rare-Word Evaluation

Build a small rare-word similarity benchmark.

## Exercise 7 — Model Comparison

Compare GloVe, Word2Vec, and subword-aware neighbors.

## Exercise 8 — Arabic Embeddings

Compare Arabic subword overlap before and after normalization or segmentation.

## Challenge Exercises

1. Implement AdaGrad for GloVe.
2. Hash character n-grams into fixed buckets.
3. Combine independent word and subword vectors.
4. Evaluate analogies by morphology category.
5. Build a reusable embedding evaluation report.

# 27. Summary and Next Lesson

In this lesson:

- GloVe used weighted global co-occurrence statistics;
- logarithmic counts formed the reconstruction target;
- NumPy training learned word and context vectors;
- nearest neighbors supported qualitative analysis;
- word-only models were shown to struggle with rare and unseen forms;
- character n-grams encoded internal and boundary patterns;
- subword vectors were combined into word vectors;
- OOV vectors were built from known n-grams;
- a toy FastText-style Skip-Gram model was implemented;
- morphological benefits and false spelling similarity were contrasted;
- intrinsic and extrinsic evaluation were separated;
- Arabic morphology and clitics motivated subword-aware representations.

## Next Lesson

**Lesson 24: Pretrained Embeddings and Embedding-Based Text Classification**
introduces loading pretrained vectors, vocabulary coverage, document-vector
construction, frozen versus trainable embeddings, and downstream classification.

# References

- Pennington, J., Socher, R., & Manning, C. D. *GloVe: Global Vectors for Word Representation*.
- Bojanowski, P. et al. *Enriching Word Vectors with Subword Information*.
- Mikolov, T. et al. *Distributed Representations of Words and Phrases and their Compositionality*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.